# PROJEKT CZ. 2: UCZENIE MASZYNOWE

In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error, recall_score, mean_absolute_error, r2_score
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import warnings

warnings.filterwarnings('ignore')

results_data = []


WCZYTANIE I PRZYGOTOWANIE DANYCH

In [ ]:


df = pd.read_csv('../../data/credit_risk_dataset.csv')

# Usuwanie anomalii
df = df[df['person_age'] < 100]
df = df[df['person_emp_length'] < 60]

# Uzupełnienie braków i kodowanie zmiennych kategorycznych
df = df.fillna(df.mean(numeric_only=True)) 
df = pd.get_dummies(df, drop_first=True)
df = df.astype(float)

# Zmienne docelowe:
#   - klasyfikacja: loan_status (0/1 - czy pożyczka spłacona)
#   - regresja:     loan_int_rate (wysokość oprocentowania)
TARGET_CLF = 'loan_status'
TARGET_REG = 'loan_int_rate'

y_clf = df[TARGET_CLF].values.ravel()
y_reg = df[TARGET_REG].values.ravel()

# Usuwamy OBIE kolumny docelowe z X (uniknięcie target leakage)
X_clf = df.drop([TARGET_CLF, TARGET_REG], axis=1).values
X_reg = df.drop([TARGET_REG, TARGET_CLF], axis=1).values

# Podział 70% train / 30% test
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.3, stratify=y_clf, random_state=42)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42)

# Skalowanie (tylko kolumny numeryczne – bez 0/1)
binary_cols = [i for i in range(X_train_clf.shape[1])
               if np.isin(X_train_clf[:, i], [0, 1]).all()]
numeric_cols = [i for i in range(X_train_clf.shape[1])
                if i not in binary_cols]

DANE

In [ ]:
df.head(5)

KLASYFIKACJA

In [ ]:
scaler_clf = StandardScaler()
X_train_clf_s = X_train_clf.copy()
X_test_clf_s = X_test_clf.copy()
X_train_clf_s[:, numeric_cols] = scaler_clf.fit_transform(X_train_clf[:, numeric_cols])
X_test_clf_s[:, numeric_cols] = scaler_clf.transform(X_test_clf[:, numeric_cols])

# ===== REGRESJA =====
binary_cols_reg = [i for i in range(X_train_reg.shape[1])
                   if np.isin(X_train_reg[:, i], [0, 1]).all()]
numeric_cols_reg = [i for i in range(X_train_reg.shape[1])
                    if i not in binary_cols_reg]

scaler_reg = StandardScaler()
X_train_reg_s = X_train_reg.copy()
X_test_reg_s = X_test_reg.copy()
X_train_reg_s[:, numeric_cols_reg] = scaler_reg.fit_transform(X_train_reg[:, numeric_cols_reg])
X_test_reg_s[:, numeric_cols_reg] = scaler_reg.transform(X_test_reg[:, numeric_cols_reg])

print("Dane przygotowane.")
print(f"  Rozmiar zbioru uczącego  (clf): {X_train_clf_s.shape}") #shape zwraca (liczba wierszy, liczba kolumn)
print(f"  Rozmiar zbioru testowego (clf): {X_test_clf_s.shape}")
print(f"  Rozmiar zbioru uczącego  (reg): {X_train_reg_s.shape}")
print(f"  Rozmiar zbioru testowego (reg): {X_test_reg_s.shape}")

POMOCNICZE FUNKCJE

In [ ]:
def print_header(title):
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")

def print_param_header(desc):
    print(f"\n--- {desc} ---")

# PROBLEM KLASYFIKACYJNY

### KNN


KNN – PARAMETR 1: liczba sąsiadów (n_neighbors)

In [ ]:
print_param_header("KNN | PARAMETR 1: liczba sąsiadów (n_neighbors)")
print(f"  {'n_neighbors':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for k in [3, 5, 7, 9]:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))  
    tr_recall = recall_score(y_train_clf, m.predict(X_train_clf_s))
    te_recall = recall_score(y_test_clf, m.predict(X_test_clf_s))
    print(f"  k = {k:<11} {tr:>12.4f} {te:>12.4f}")
    print(f"  (Recall) {'':<7} {tr_recall:>12.4f} {te_recall:>12.4f}")

    results_data.append({
        'Model': 'KNN',
        'Parametr': f'n_neighbors={k}',
        'Accuracy': round(te, 4),
        'Recall_klasa_1': round(te_recall, 4)
    })

KNN – PARAMETR 2: wagi sąsiadów (weights)

In [ ]:
print_param_header("KNN | PARAMETR 2: metryka odległości (metric)")
print(f"  {'metric':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for metr in ['euclidean', 'manhattan', 'chebyshev', 'minkowski']:
    m = KNeighborsClassifier(n_neighbors=5, metric=metr)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    tr_recall = recall_score(y_train_clf, m.predict(X_train_clf_s))
    te_recall = recall_score(y_test_clf, m.predict(X_test_clf_s))
    print(f"  {metr:<15} {tr:>12.4f} {te:>12.4f}")
    print(f"  (Recall) {'':<7} {tr_recall:>12.4f} {te_recall:>12.4f}")
    results_data.append({
        'Model': 'KNN',
        'Parametr': f'metric={metr}',
        'Accuracy': round(te, 4),
        'Recall_klasa_1': round(te_recall, 4)
    })


### SVM


SVM – PARAMETR 1: parametr regularyzacji C

In [ ]:
print_param_header("SVM | PARAMETR 1: parametr regularyzacji (C)")
print(f"  {'C':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for c in [0.1, 1.0, 10.0, 100.0]:
    m = SVC(C=c, kernel='rbf', random_state=42)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    tr_recall = recall_score(y_train_clf, m.predict(X_train_clf_s))
    te_recall = recall_score(y_test_clf, m.predict(X_test_clf_s))
    print(f"  C = {c:<11} {tr:>12.4f} {te:>12.4f}")
    print(f"  (Recall) {'':<7} {tr_recall:>12.4f} {te_recall:>12.4f}")
    results_data.append({
        'Model': 'SVM',
        'Parametr': f'C={c}',
        'Accuracy': round(te, 4),
        'Recall_klasa_1': round(te_recall, 4)
    })


SVR – PARAMETR 2: rodzaj jądra (kernel)

In [ ]:
print_param_header("SVM | PARAMETR 2: rodzaj jądra (kernel)")
print(f"  {'kernel':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for kern in ['linear', 'poly', 'rbf', 'sigmoid']:
    m = SVC(C=1.0, kernel=kern, random_state=42)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    tr_recall = recall_score(y_train_clf, m.predict(X_train_clf_s))
    te_recall = recall_score(y_test_clf, m.predict(X_test_clf_s))
    print(f"  {kern:<15} {tr:>12.4f} {te:>12.4f}")
    print(f"  (Recall) {'':<7} {tr_recall:>12.4f} {te_recall:>12.4f}")
    results_data.append({
        'Model': 'SVM',
        'Parametr': f'kernel={kern}',
        'Accuracy': round(te, 4),
        'Recall_klasa_1': round(te_recall, 4)
    })




### DRZEWO DECYZYJNE

DRZEWO – PARAMETR 1: maksymalna głębokość (max_depth)

In [ ]:
print_param_header("Drzewo Decyzyjne | PARAMETR 1: maksymalna głębokość (max_depth)")
print(f"  {'max_depth':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for depth in [3, 5, 10, None]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    tr_recall = recall_score(y_train_clf, m.predict(X_train_clf_s))
    te_recall = recall_score(y_test_clf, m.predict(X_test_clf_s))
    label = str(depth) if depth is not None else 'None (brak)'
    print(f"  {label:<15} {tr:>12.4f} {te:>12.4f}")
    print(f"  (Recall) {'':<7} {tr_recall:>12.4f} {te_recall:>12.4f}")
    results_data.append({
        'Model': 'Decision Tree',
        'Parametr': f'max_depth={depth}',
        'Accuracy': round(te, 4),
        'Recall_klasa_1': round(te_recall, 4)
    })


DRZEWO – PARAMETR 2: min. liczba obserwacji w liściu (min_samples_leaf)

In [ ]:
print_param_header("Drzewo Decyzyjne | PARAMETR 2: min. liczba obserwacji w liściu (min_samples_leaf)")
print(f"  {'min_samples_leaf':<20} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*20} {'-'*12} {'-'*12}")
for msl in [1, 5, 20, 50]:
    m = DecisionTreeClassifier(min_samples_leaf=msl, random_state=42)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    tr_recall = recall_score(y_train_clf, m.predict(X_train_clf_s))
    te_recall = recall_score(y_test_clf, m.predict(X_test_clf_s))
    print(f"  {msl:<20} {tr:>12.4f} {te:>12.4f}")
    print(f"  (Recall) {'':<7} {tr_recall:>12.4f} {te_recall:>12.4f}")
    results_data.append({
        'Model': 'Decision Tree',
        'Parametr': f'min_samples_leaf={msl}',
        'Accuracy': round(te, 4),
        'Recall_klasa_1': round(te_recall, 4)
    })


### LAS LOSOWY

LAS LOSOWY – PARAMETR 1: liczba drzew (n_estimators)

In [ ]:
print_param_header("Las Losowy | PARAMETR 1: liczba drzew (n_estimators)")
print(f"  {'n_estimators':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for n_est in [10, 50, 100, 200]:
    m = RandomForestClassifier(n_estimators=n_est, random_state=42, n_jobs=-1)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    tr_recall = recall_score(y_train_clf, m.predict(X_train_clf_s))
    te_recall = recall_score(y_test_clf, m.predict(X_test_clf_s))
    print(f"  {n_est:<15} {tr:>12.4f} {te:>12.4f}")
    print(f"  (Recall) {'':<7} {tr_recall:>12.4f} {te_recall:>12.4f}")
    results_data.append({
        'Model': 'Random Forest',
        'Parametr': f'n_estimators={n_est}',
        'Accuracy': round(te, 4),
        'Recall_klasa_1': round(te_recall, 4)
    })


LAS LOSOWY – PARAMETR 2: maksymalna głębokość (max_depth)

In [ ]:
print_param_header("Las Losowy | PARAMETR 2: maksymalna głębokość drzew (max_depth)")
print(f"  {'max_depth':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for depth in [3, 5, 10, None]:
    m = RandomForestClassifier(n_estimators=100, max_depth=depth,
                               random_state=42, n_jobs=-1)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    tr_recall = recall_score(y_train_clf, m.predict(X_train_clf_s))
    te_recall = recall_score(y_test_clf, m.predict(X_test_clf_s))
    label = str(depth) if depth is not None else 'None (brak)'
    print(f"  {label:<15} {tr:>12.4f} {te:>12.4f}")
    print(f"  (Recall) {'':<7} {tr_recall:>12.4f} {te_recall:>12.4f}")
    results_data.append({
        'Model': 'Random Forest',
        'Parametr': f'max_depth={depth}',
        'Accuracy': round(te, 4),
        'Recall_klasa_1': round(te_recall, 4)
    })



# PROBLEM REGRESYJNY

### KNN

PARAMETR 1: liczba sąsiadów (n_neighbors)

In [ ]:
print_param_header("KNN | PARAMETR 1: liczba sąsiadów (n_neighbors)")
print(f"  {'n_neighbors':<12} {'RMSE_train':>12} {'RMSE_test':>12} {'MAE_train':>12} {'MAE_test':>12} {'R2_train':>12} {'R2_test':>12}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for k in [3, 5, 7, 9]:
    m = KNeighborsRegressor(n_neighbors=k)
    m.fit(X_train_reg_s, y_train_reg)
    #comment
    y_tr = m.predict(X_train_reg_s)
    y_te = m.predict(X_test_reg_s)
    
    rmse_tr = np.sqrt(mean_squared_error(y_train_reg, y_tr))
    rmse_te = np.sqrt(mean_squared_error(y_test_reg, y_te))
    mae_tr = mean_absolute_error(y_train_reg, y_tr)
    mae_te = mean_absolute_error(y_test_reg, y_te)
    r2_tr = r2_score(y_train_reg, y_tr)
    r2_te = r2_score(y_test_reg, y_te)
    
    print(f"  k = {k:<8} {rmse_tr:>12.4f} {rmse_te:>12.4f} {mae_tr:>12.4f} {mae_te:>12.4f} {r2_tr:>12.4f} {r2_te:>12.4f}")

    wyniki.append({
        'model': 'KNN',
        'parametr': 'n_neighbors',
        'wartosc': k,
        'RMSE_train': rmse_tr,
        'RMSE_test': rmse_te,
        'MAE_train': mae_tr,
        'MAE_test': mae_te,
        'R2_train': r2_tr,
        'R2_test': r2_te
    })

PARAMETR 2: metryka odległości (metric)

In [ ]:
print_param_header("KNN | PARAMETR 2: metryka odległości (metric)")
print(f"  {'metric':<12} {'RMSE_train':>12} {'RMSE_test':>12} {'MAE_train':>12} {'MAE_test':>12} {'R2_train':>12} {'R2_test':>12}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for metr in ['euclidean', 'manhattan', 'chebyshev', 'minkowski']:
    m = KNeighborsRegressor(n_neighbors=5, metric=metr)
    m.fit(X_train_reg_s, y_train_reg)
    
    y_tr = m.predict(X_train_reg_s)
    y_te = m.predict(X_test_reg_s)
    
    rmse_tr = np.sqrt(mean_squared_error(y_train_reg, y_tr))
    rmse_te = np.sqrt(mean_squared_error(y_test_reg, y_te))
    mae_tr = mean_absolute_error(y_train_reg, y_tr)
    mae_te = mean_absolute_error(y_test_reg, y_te)
    r2_tr = r2_score(y_train_reg, y_tr)
    r2_te = r2_score(y_test_reg, y_te)
    
    print(f"  {metr:<12} {rmse_tr:>12.4f} {rmse_te:>12.4f} {mae_tr:>12.4f} {mae_te:>12.4f} {r2_tr:>12.4f} {r2_te:>12.4f}")

    wyniki.append({
        'model': 'KNN',
        'parametr': 'metric',
        'wartosc': metr,
        'RMSE_train': rmse_tr,
        'RMSE_test': rmse_te,
        'MAE_train': mae_tr,
        'MAE_test': mae_te,
        'R2_train': r2_tr,
        'R2_test': r2_te
    })

### SVR

PARAMETR 1: parametr regularyzacji C

In [ ]:
print_param_header("SVR | PARAMETR 1: parametr regularyzacji (C)")
print(f"  {'C':<12} {'RMSE_train':>12} {'RMSE_test':>12} {'MAE_train':>12} {'MAE_test':>12} {'R2_train':>12} {'R2_test':>12}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for c in [0.1, 1.0, 10.0, 100.0]:
    m = SVR(C=c, kernel='rbf')
    m.fit(X_train_reg_s, y_train_reg)
    
    y_tr = m.predict(X_train_reg_s)
    y_te = m.predict(X_test_reg_s)
    
    rmse_tr = np.sqrt(mean_squared_error(y_train_reg, y_tr))
    rmse_te = np.sqrt(mean_squared_error(y_test_reg, y_te))
    mae_tr = mean_absolute_error(y_train_reg, y_tr)
    mae_te = mean_absolute_error(y_test_reg, y_te)
    r2_tr = r2_score(y_train_reg, y_tr)
    r2_te = r2_score(y_test_reg, y_te)
    
    print(f"  {c:<12} {rmse_tr:>12.4f} {rmse_te:>12.4f} {mae_tr:>12.4f} {mae_te:>12.4f} {r2_tr:>12.4f} {r2_te:>12.4f}")

    wyniki.append({
        'model': 'SVR',
        'parametr': 'C',
        'wartosc': c,
        'RMSE_train': rmse_tr,
        'RMSE_test': rmse_te,
        'MAE_train': mae_tr,
        'MAE_test': mae_te,
        'R2_train': r2_tr,
        'R2_test': r2_te
    })

PARAMETR 2: rodzaj jądra (kernel)

In [ ]:
print_param_header("SVR | PARAMETR 2: rodzaj jądra (kernel)")
print(f"  {'kernel':<12} {'RMSE_train':>12} {'RMSE_test':>12} {'MAE_train':>12} {'MAE_test':>12} {'R2_train':>12} {'R2_test':>12}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for kern in ['linear', 'poly', 'rbf', 'sigmoid']:
    m = SVR(C=1.0, kernel=kern)
    m.fit(X_train_reg_s, y_train_reg)
    
    y_tr = m.predict(X_train_reg_s)
    y_te = m.predict(X_test_reg_s)
    ###
    rmse_tr = np.sqrt(mean_squared_error(y_train_reg, y_tr))
    rmse_te = np.sqrt(mean_squared_error(y_test_reg, y_te))
    mae_tr = mean_absolute_error(y_train_reg, y_tr)
    mae_te = mean_absolute_error(y_test_reg, y_te)
    r2_tr = r2_score(y_train_reg, y_tr)
    r2_te = r2_score(y_test_reg, y_te)
    
    print(f"  {kern:<12} {rmse_tr:>12.4f} {rmse_te:>12.4f} {mae_tr:>12.4f} {mae_te:>12.4f} {r2_tr:>12.4f} {r2_te:>12.4f}")

    wyniki.append({
        'model': 'SVR',
        'parametr': 'kernel',
        'wartosc': kern,
        'RMSE_train': rmse_tr,
        'RMSE_test': rmse_te,
        'MAE_train': mae_tr,
        'MAE_test': mae_te,
        'R2_train': r2_tr,
        'R2_test': r2_te
    })

### DRZEWO DECYZYJNE

PARAMETR 1: maksymalna głębokość (max_depth)

In [ ]:
print_param_header("Drzewo Decyzyjne | PARAMETR 1: maksymalna głębokość (max_depth)")
print(f"  {'max_depth':<12} {'RMSE_train':>12} {'RMSE_test':>12} {'MAE_train':>12} {'MAE_test':>12} {'R2_train':>12} {'R2_test':>12}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for depth in [3, 5, 10, None]:
    m = DecisionTreeRegressor(max_depth=depth, random_state=42)
    m.fit(X_train_reg_s, y_train_reg)
    
    y_tr = m.predict(X_train_reg_s)
    y_te = m.predict(X_test_reg_s)
    
    rmse_tr = np.sqrt(mean_squared_error(y_train_reg, y_tr))
    rmse_te = np.sqrt(mean_squared_error(y_test_reg, y_te))
    mae_tr = mean_absolute_error(y_train_reg, y_tr)
    mae_te = mean_absolute_error(y_test_reg, y_te)
    r2_tr = r2_score(y_train_reg, y_tr)
    r2_te = r2_score(y_test_reg, y_te)
    
    label = str(depth) if depth is not None else 'None'
    print(f"  {label:<12} {rmse_tr:>12.4f} {rmse_te:>12.4f} {mae_tr:>12.4f} {mae_te:>12.4f} {r2_tr:>12.4f} {r2_te:>12.4f}")

    wyniki.append({
        'model': 'Decision Tree',
        'parametr': 'max_depth',
        'wartosc': str(depth),
        'RMSE_train': rmse_tr,
        'RMSE_test': rmse_te,
        'MAE_train': mae_tr,
        'MAE_test': mae_te,
        'R2_train': r2_tr,
        'R2_test': r2_te
    })

PARAMETR 2: min. liczba obserwacji w liściu (min_samples_leaf)

In [ ]:
print_param_header("Drzewo Decyzyjne | PARAMETR 2: min. liczba obserwacji w liściu (min_samples_leaf)")
print(f"  {'min_samp_leaf':<12} {'RMSE_train':>12} {'RMSE_test':>12} {'MAE_train':>12} {'MAE_test':>12} {'R2_train':>12} {'R2_test':>12}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for msl in [1, 5, 20, 50]:
    m = DecisionTreeRegressor(min_samples_leaf=msl, random_state=42)
    m.fit(X_train_reg_s, y_train_reg)
    
    y_tr = m.predict(X_train_reg_s)
    y_te = m.predict(X_test_reg_s)
    
    rmse_tr = np.sqrt(mean_squared_error(y_train_reg, y_tr))
    rmse_te = np.sqrt(mean_squared_error(y_test_reg, y_te))
    mae_tr = mean_absolute_error(y_train_reg, y_tr)
    mae_te = mean_absolute_error(y_test_reg, y_te)
    r2_tr = r2_score(y_train_reg, y_tr)
    r2_te = r2_score(y_test_reg, y_te)
    
    print(f"  {msl:<12} {rmse_tr:>12.4f} {rmse_te:>12.4f} {mae_tr:>12.4f} {mae_te:>12.4f} {r2_tr:>12.4f} {r2_te:>12.4f}")

    wyniki.append({
        'model': 'Decision Tree',
        'parametr': 'min_samples_leaf',
        'wartosc': msl,
        'RMSE_train': rmse_tr,
        'RMSE_test': rmse_te,
        'MAE_train': mae_tr,
        'MAE_test': mae_te,
        'R2_train': r2_tr,
        'R2_test': r2_te
    })

### LAS LOSOWY

PARAMETR 1: liczba drzew (n_estimators)

In [ ]:
print_param_header("Las Losowy | PARAMETR 1: liczba drzew (n_estimators)")
print(f"  {'n_estimators':<12} {'RMSE_train':>12} {'RMSE_test':>12} {'MAE_train':>12} {'MAE_test':>12} {'R2_train':>12} {'R2_test':>12}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for n_est in [10, 50, 100, 200]:
    m = RandomForestRegressor(n_estimators=n_est, random_state=42, n_jobs=-1)
    m.fit(X_train_reg_s, y_train_reg)
    
    y_tr = m.predict(X_train_reg_s)
    y_te = m.predict(X_test_reg_s)
    
    rmse_tr = np.sqrt(mean_squared_error(y_train_reg, y_tr))
    rmse_te = np.sqrt(mean_squared_error(y_test_reg, y_te))
    mae_tr = mean_absolute_error(y_train_reg, y_tr)
    mae_te = mean_absolute_error(y_test_reg, y_te)
    r2_tr = r2_score(y_train_reg, y_tr)
    r2_te = r2_score(y_test_reg, y_te)
    
    print(f"  {n_est:<12} {rmse_tr:>12.4f} {rmse_te:>12.4f} {mae_tr:>12.4f} {mae_te:>12.4f} {r2_tr:>12.4f} {r2_te:>12.4f}")

    wyniki.append({
        'model': 'Random Forest',
        'parametr': 'n_estimators',
        'wartosc': n_est,
        'RMSE_train': rmse_tr,
        'RMSE_test': rmse_te,
        'MAE_train': mae_tr,
        'MAE_test': mae_te,
        'R2_train': r2_tr,
        'R2_test': r2_te
    })

PARAMETR 2: maksymalna głębokość (max_depth)

In [ ]:
print_param_header("Las Losowy | PARAMETR 2: maksymalna głębokość drzew (max_depth)")
print(f"  {'max_depth':<12} {'RMSE_train':>12} {'RMSE_test':>12} {'MAE_train':>12} {'MAE_test':>12} {'R2_train':>12} {'R2_test':>12}")
print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for depth in [3, 5, 10, None]:
    m = RandomForestRegressor(n_estimators=100, max_depth=depth, random_state=42, n_jobs=-1)
    m.fit(X_train_reg_s, y_train_reg)
    
    y_tr = m.predict(X_train_reg_s)
    y_te = m.predict(X_test_reg_s)
    
    rmse_tr = np.sqrt(mean_squared_error(y_train_reg, y_tr))
    rmse_te = np.sqrt(mean_squared_error(y_test_reg, y_te))
    mae_tr = mean_absolute_error(y_train_reg, y_tr)
    mae_te = mean_absolute_error(y_test_reg, y_te)
    r2_tr = r2_score(y_train_reg, y_tr)
    r2_te = r2_score(y_test_reg, y_te)
    
    label = str(depth) if depth is not None else 'None'
    print(f"  {label:<12} {rmse_tr:>12.4f} {rmse_te:>12.4f} {mae_tr:>12.4f} {mae_te:>12.4f} {r2_tr:>12.4f} {r2_te:>12.4f}")

    wyniki.append({
        'model': 'Random Forest',
        'parametr': 'max_depth',
        'wartosc': str(depth),
        'RMSE_train': rmse_tr,
        'RMSE_test': rmse_te,
        'MAE_train': mae_tr,
        'MAE_test': mae_te,
        'R2_train': r2_tr,
        'R2_test': r2_te
    })

In [ ]:
df_wyniki = pd.DataFrame(wyniki)
df_wyniki.to_excel('wyniki_regresja.xlsx', index=False)